# Manual EV trip-energy prediction

Edit the values in Section 2, then run all cells. The example is prefilled for **Route El Ain Km 4 -> ENET'Com, Sfax** using the compact route result: about 12.2 km and 22.2 minutes.

> Important: the model was trained on row-level total energy, not on additive road segments. The notebook displays both a segment sum and one aggregate-trip prediction because summing segment predictions can repeat the model intercept and overestimate energy.

## 1. Load the saved model

In [1]:
from pathlib import Path
import json

import joblib
import pandas as pd
from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'ev_energy_kaggle':
    KAGGLE_ROOT = PROJECT_ROOT
elif (PROJECT_ROOT / 'ev_energy_kaggle').is_dir():
    KAGGLE_ROOT = PROJECT_ROOT / 'ev_energy_kaggle'
else:
    raise FileNotFoundError('Run this notebook from the project root or ev_energy_kaggle folder.')

OUTPUT_DIR = KAGGLE_ROOT / 'Main' / 'output'
MODEL_PATH = OUTPUT_DIR / 'best_energy_model.joblib'
METADATA_PATH = OUTPUT_DIR / 'best_energy_model_metadata.json'

model = joblib.load(MODEL_PATH)
metadata = json.loads(METADATA_PATH.read_text(encoding='utf-8'))
FEATURES = metadata['selected_features']
print('Model:', metadata['model_name'])
print('Target:', metadata['target'])
print('Feature count:', len(FEATURES))

Model: LinearRegression
Target: Energy_Consumption_kWh
Feature count: 12


## 2. Enter trip values here

Change these values for another vehicle state, weather observation, or route. Distances and speeds below come from the compact Ain Km 4 -> ENET'Com route result.

In [2]:
# Vehicle and destination-weather values shared by every route section.
BATTERY_SOC_PERCENT = 80.0
BATTERY_VOLTAGE_V = 352.0          # BMW i3 nominal traction voltage
BATTERY_TEMPERATURE_C = 25.0
AMBIENT_TEMPERATURE_C = 21.5
HUMIDITY_PERCENT = 57.0
WIND_SPEED_KMH = 8.6
TIRE_PRESSURE_PSI = 32.0           # Mean of four tires
AVERAGE_SLOPE_PERCENT = 0.0        # Current route-grade assumption
AVERAGE_ACCELERATION_MPS2 = 0.0    # Steady-speed assumption

# BMW i3 / feature-engineering constants used during training.
VEHICLE_MASS_KG = 1270.0
GRAVITY_MPS2 = 9.81
ROLLING_RESISTANCE = 0.012

# Approximate compact route sections returned for Ain Km 4 -> ENET'Com.
ROUTE_SEGMENTS = [
    {'road_type': 'urban',    'distance_km': 8.760, 'speed_kmh': 31.3},
    {'road_type': 'suburban', 'distance_km': 0.246, 'speed_kmh': 77.0},
    {'road_type': 'highway',  'distance_km': 3.192, 'speed_kmh': 48.0},
]

print('Configured route distance:', round(sum(s['distance_km'] for s in ROUTE_SEGMENTS), 3), 'km')

Configured route distance: 12.198 km


## 3. Build the exact model features

In [3]:
from math import atan, sin

slope_ratio = AVERAGE_SLOPE_PERCENT / 100.0
slope_force = VEHICLE_MASS_KG * GRAVITY_MPS2 * sin(atan(slope_ratio))
inertial_force = VEHICLE_MASS_KG * AVERAGE_ACCELERATION_MPS2
rolling_force = ROLLING_RESISTANCE * VEHICLE_MASS_KG * GRAVITY_MPS2

common_features = {
    'Battery_State_%': BATTERY_SOC_PERCENT,
    'Battery_Voltage_V': BATTERY_VOLTAGE_V,
    'Battery_Temperature_C': BATTERY_TEMPERATURE_C,
    'Temperature_C': AMBIENT_TEMPERATURE_C,
    'Humidity_%': HUMIDITY_PERCENT,
    'Wind_Speed_ms': WIND_SPEED_KMH / 3.6,
    'Tire_Pressure_psi': TIRE_PRESSURE_PSI,
    'slope_force_proxy_N': slope_force,
    'inertial_force_proxy_N': inertial_force,
    'rolling_force_assumed_N': rolling_force,
}

feature_rows = []
for segment in ROUTE_SEGMENTS:
    feature_rows.append({
        **common_features,
        'Distance_Travelled_km': segment['distance_km'],
        'Speed_kmh': segment['speed_kmh'],
    })

segment_features_df = pd.DataFrame(feature_rows, columns=FEATURES)
display(Markdown('### Exact per-segment model inputs'))
display(segment_features_df.round(3))
assert list(segment_features_df.columns) == FEATURES
assert segment_features_df.notna().all().all()

### Exact per-segment model inputs

,Battery_State_%,Battery_Voltage_V,Battery_Temperature_C,Temperature_C,Humidity_%,Wind_Speed_ms,Tire_Pressure_psi,Distance_Travelled_km,slope_force_proxy_N,Speed_kmh,inertial_force_proxy_N,rolling_force_assumed_N
0,80.0,352.0,25.0,21.5,57.0,2.389,32.0,8.760,0.0,31.3,0.0,149.504
1,80.0,352.0,25.0,21.5,57.0,2.389,32.0,0.246,0.0,77.0,0.0,149.504
2,80.0,352.0,25.0,21.5,57.0,2.389,32.0,3.192,0.0,48.0,0.0,149.504


## 4. Predict energy

In [4]:
segment_predictions = model.predict(segment_features_df)
segment_results = pd.DataFrame(ROUTE_SEGMENTS).copy()
segment_results['predicted_energy_kwh'] = segment_predictions
display(Markdown('### Per-segment predictions'))
display(segment_results.round(3))

segment_sum_kwh = float(segment_predictions.sum())
total_distance_km = float(segment_results['distance_km'].sum())
weighted_speed_kmh = float(
    (segment_results['distance_km'] * segment_results['speed_kmh']).sum()
    / total_distance_km
)

aggregate_features = {
    **common_features,
    'Distance_Travelled_km': total_distance_km,
    'Speed_kmh': weighted_speed_kmh,
}
aggregate_df = pd.DataFrame([aggregate_features], columns=FEATURES)
aggregate_prediction_kwh = float(model.predict(aggregate_df)[0])

summary = pd.DataFrame([
    {'method': 'Sum of segment predictions (experimental)', 'energy_kwh': segment_sum_kwh},
    {'method': 'One aggregate trip row (diagnostic)', 'energy_kwh': aggregate_prediction_kwh},
])
display(Markdown('### Trip prediction comparison'))
display(summary.round(3))
print('Total distance:', round(total_distance_km, 3), 'km')
print('Distance-weighted speed:', round(weighted_speed_kmh, 2), 'km/h')
print('Segment sum:', round(segment_sum_kwh, 3), 'kWh')
print('Aggregate-row prediction:', round(aggregate_prediction_kwh, 3), 'kWh')

### Per-segment predictions

,road_type,distance_km,speed_kmh,predicted_energy_kwh
0,urban,8.760,31.3,5.186
1,suburban,0.246,77.0,6.861
2,highway,3.192,48.0,5.626


### Trip prediction comparison

,method,energy_kwh
0,Sum of segment predictions (experimental),17.672
1,One aggregate trip row (diagnostic),5.690


Total distance: 12.198 km
Distance-weighted speed: 36.59 km/h
Segment sum: 17.672 kWh
Aggregate-row prediction: 5.69 kWh


## 5. Predict the entire trip as one row (no segments)

This prediction sends exactly one feature row to the model. Edit the total distance and average speed below when testing another trip.

In [5]:
# Ain Km 4 -> ENET'Com whole-trip inputs (no route segments).
ENTIRE_TRIP_DISTANCE_KM = 12.198
ENTIRE_TRIP_AVERAGE_SPEED_KMH = 36.59

entire_trip_features = {
    **common_features,
    'Distance_Travelled_km': ENTIRE_TRIP_DISTANCE_KM,
    'Speed_kmh': ENTIRE_TRIP_AVERAGE_SPEED_KMH,
}
entire_trip_df = pd.DataFrame([entire_trip_features], columns=FEATURES)
entire_trip_energy_kwh = float(model.predict(entire_trip_df)[0])

display(Markdown('### One-row input sent to the model'))
display(entire_trip_df.round(3))
print('Predicted energy for the entire trip:', round(entire_trip_energy_kwh, 3), 'kWh')
print(
    'Predicted consumption:',
    round(entire_trip_energy_kwh / ENTIRE_TRIP_DISTANCE_KM * 100, 2),
    'kWh/100 km',
)

### One-row input sent to the model

,Battery_State_%,Battery_Voltage_V,Battery_Temperature_C,Temperature_C,Humidity_%,Wind_Speed_ms,Tire_Pressure_psi,Distance_Travelled_km,slope_force_proxy_N,Speed_kmh,inertial_force_proxy_N,rolling_force_assumed_N
0,80.0,352.0,25.0,21.5,57.0,2.389,32.0,12.198,0.0,36.59,0.0,149.504


Predicted energy for the entire trip: 5.69 kWh
Predicted consumption: 46.64 kWh/100 km


## 6. Interpretation

For the prefilled values, the current model gives approximately **17.67 kWh** when three section predictions are summed, versus **5.69 kWh** for one aggregate trip row. This large disagreement is evidence that the trained row-level model is not reliably additive across route segments. Treat both as experimental until the model is retrained with segment-level energy targets or a validated trip-level aggregation method.